In the [previous post](2025-12-23-sam3-gemini-segmentation.html), we explored SAM3's image segmentation with text and box prompts. Now let's use SAM3 for **video tracking** - following a tennis ball across frames using just a text prompt.

## Setup

In [ ]:
# SAM3 requires transformers from source
# !pip install -q git+https://github.com/huggingface/transformers.git
# !pip install -q torch pillow requests matplotlib accelerate

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from transformers import Sam3VideoModel, Sam3VideoProcessor
from transformers.video_utils import load_video

device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
dtype = torch.float16 if device == "cuda" else torch.float32

model = Sam3VideoModel.from_pretrained("facebook/sam3").to(device, dtype=dtype)
processor = Sam3VideoProcessor.from_pretrained("facebook/sam3")

%config InlineBackend.figure_format = 'retina'

## Load Video

A short tennis clip - we'll track the ball across frames.

In [ ]:
# Download a short tennis video
import requests
from pathlib import Path

video_url = "https://storage.googleapis.com/gtv-videos-bucket/sample/ForBiggerMeltdowns.mp4"
video_path = Path("tennis_clip.mp4")

# Use a tennis video from Pexels (free stock)
video_url = "https://videos.pexels.com/video-files/2519660/2519660-sd_640_360_30fps.mp4"

if not video_path.exists():
    r = requests.get(video_url)
    video_path.write_bytes(r.content)
    
print(f"Video saved: {video_path}")

In [ ]:
# Load video frames
video_frames, fps = load_video(str(video_path))
print(f"Loaded {len(video_frames)} frames at {fps} fps")

# Show first frame
plt.figure(figsize=(10, 6))
plt.imshow(video_frames[0])
plt.title("First Frame")
plt.axis('off')
plt.show()

## Track with Text Prompt

Use SAM3's video mode to track "tennis ball" across all frames.

In [ ]:
# Initialize video session
inference_session = processor.init_video_session(
    video=video_frames,
    inference_device=device,
    processing_device="cpu",
    video_storage_device="cpu",
    dtype=dtype,
)

# Add text prompt
inference_session = processor.add_text_prompt(
    inference_session=inference_session,
    text="tennis ball",
)

print("Session initialized with prompt: 'tennis ball'")

In [ ]:
# Track through all frames
outputs_per_frame = {}

for model_outputs in model.propagate_in_video_iterator(
    inference_session=inference_session,
    max_frame_num_to_track=len(video_frames)
):
    processed = processor.postprocess_outputs(inference_session, model_outputs)
    outputs_per_frame[model_outputs.frame_idx] = processed

print(f"Tracked {len(outputs_per_frame)} frames")

## Visualize Tracking

In [ ]:
def overlay_mask(frame, mask, color=(255, 100, 100), alpha=0.5):
    """Overlay mask on frame."""
    frame = np.array(frame).astype(float)
    mask = mask.cpu().numpy() if torch.is_tensor(mask) else mask
    
    for c in range(3):
        frame[:,:,c] = np.where(mask, frame[:,:,c]*(1-alpha) + color[c]*alpha, frame[:,:,c])
    
    return frame.astype(np.uint8)

# Show tracking on key frames
key_frames = [0, len(video_frames)//4, len(video_frames)//2, 3*len(video_frames)//4, len(video_frames)-1]
key_frames = [f for f in key_frames if f in outputs_per_frame]

fig, axes = plt.subplots(1, len(key_frames), figsize=(4*len(key_frames), 4))

for i, frame_idx in enumerate(key_frames):
    frame = video_frames[frame_idx]
    output = outputs_per_frame[frame_idx]
    
    if output['masks'] is not None and len(output['masks']) > 0:
        mask = output['masks'][0]  # First object
        frame = overlay_mask(frame, mask)
    
    axes[i].imshow(frame)
    axes[i].set_title(f"Frame {frame_idx}")
    axes[i].axis('off')

plt.suptitle("Tennis Ball Tracking with SAM3", fontweight='bold')
plt.tight_layout()
plt.show()

## Save Tracked Video

In [ ]:
import cv2

# Create output video
output_path = "tennis_tracked.mp4"
h, w = video_frames[0].shape[:2]
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(output_path, fourcc, fps, (w, h))

for frame_idx, frame in enumerate(video_frames):
    if frame_idx in outputs_per_frame:
        output = outputs_per_frame[frame_idx]
        if output['masks'] is not None and len(output['masks']) > 0:
            frame = overlay_mask(frame, output['masks'][0], color=(0, 255, 0))
    
    out.write(cv2.cvtColor(frame, cv2.COLOR_RGB2BGR))

out.release()
print(f"Saved: {output_path}")

In [ ]:
from IPython.display import Video
Video(output_path, embed=True, width=640)

## Summary

SAM3's video mode enables:
- **Text-based tracking**: Just describe what to track
- **Frame propagation**: Automatically follows object across frames
- **No manual annotation**: Unlike SAM2 which needed point/box prompts per frame

## References

- [Previous post: SAM3 Image Segmentation](2025-12-23-sam3-gemini-segmentation.html)
- [SAM3 Video on Hugging Face](https://huggingface.co/docs/transformers/main/model_doc/sam3_video)
- [SAM3 GitHub](https://github.com/facebookresearch/sam3)